In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
# Load a Dataset 

df = pd.read_csv('diamonds.csv')

In [ ]:
# view data 

df.head()


In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns


In [ ]:
# basic info 

df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df['cut'] = df['cut'].astype('category')
df['color'] = df['color'].astype('category')
df['clarity'] = df['clarity'].astype('category')

In [ ]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
df.dtypes

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols: 
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Box plot of {col}")
    plt.show()

In [ ]:
for col in num_cols: 
    Q1 = df[col].quantile(0.25)
    Q3= df[col].quantile(0.75)
    IQR = Q3 - Q1 

    lower = Q1 - 1.5 * IQR 
    upper = Q3 + 1.5 * IQR 

    outliers = df[(df[col] < lower) | (df[col] >  upper)] 

    print(f"{col} : {len(outliers)} outliers")

In [ ]:
for col in num_cols: 
    print(f"{df[col].skew()} ")

In [ ]:
df['price'] = np.log1p(df['price'])
df['carat'] = np.log1p(df['carat'])

In [ ]:
for col in ['x', 'y', 'z', 'table']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower,
                       np.where(df[col] > upper, upper, df[col]))

In [ ]:
for col in num_cols: 
    Q1 = df[col].quantile(0.25)
    Q3= df[col].quantile(0.75)
    IQR = Q3 - Q1 

    lower = Q1 - 1.5 * IQR 
    upper = Q3 + 1.5 * IQR 

    outliers = df[(df[col] < lower) | (df[col] >  upper)] 

    print(f"{col} : {len(outliers)} outliers")

In [ ]:
df['volume'] = df['x'] * df['y'] * df['z']

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True)

# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
df.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("R2 Score:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

In [ ]:
pred_actual = np.expm1(pred)
y_actual = np.expm1(y_test)

In [ ]:
results = pd.DataFrame({
    'Actual Price': y_actual,
    'Predicted Price': pred_actual
})

results.head()

In [ ]:
import joblib

joblib.dump(rf, 'diamond_price_model.pkl')

In [ ]:
df.to_excel(r'D:\diamonds_processed.xlsx', index=False)

In [ ]:
new_data = pd.DataFrame({
    'carat': [0.5],
    'depth': [61.5],
    'table': [55],
    'x': [5.1],
    'y': [5.2],
    'z': [3.1],
    'cut': ['Premium'],
    'color': ['G'],
    'clarity': ['VS1']
})

In [ ]:
new_data['carat'] = np.log1p(new_data['carat'])

In [ ]:
new_data['volume'] = (
    new_data['x'] *
    new_data['y'] *
    new_data['z']
)

In [ ]:
new_data = pd.get_dummies(new_data)

In [ ]:
new_data = new_data.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [ ]:
pred = rf.predict(new_data)

In [ ]:
final_price = np.expm1(pred)

In [ ]:
print("Predicted Diamond Price:", final_price[0])